# DOTA Pipeline (Colab)

This notebook mirrors the VisDrone Colab flow:
1. Mount Google Drive.
2. Copy dataset into `MyDrive` (idempotent).
3. Optionally extract ZIP archives if needed.
4. Build runtime config.
5. Run smoke pipeline.
6. Optionally run training/eval.


In [1]:
# Step 0: install dependencies in Colab runtime.
%pip install -q ultralytics pycocotools albumentations pyyaml tqdm


In [2]:
# Step 1: mount Google Drive and define paths.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/small_objects_auto_aug')
DRIVE_ROOT = Path('/content/drive/MyDrive')

DATASET_NAME = 'DOTA'
TARGET_DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'dota_yolo'
TARGET_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)

# Source can be:
# - prepared YOLO dataset folder in Drive (recommended)
# - zip archives folder in Drive (optional unzip step below)
SOURCE_DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'incoming' / 'dota_yolo'
SOURCE_ZIP_DIR = DRIVE_ROOT / 'datasets' / f'{DATASET_NAME.lower()}_zips'

# Unified experiment paths in Drive.
DOTA_EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments' / 'small_objects_auto_aug' / DATASET_NAME.lower()
DOTA_EXPERIMENT_RUNS = DOTA_EXPERIMENT_ROOT / 'runs'
DOTA_EXPERIMENT_ARTIFACTS = DOTA_EXPERIMENT_ROOT / 'artifacts'
SYNC_ARTIFACTS_TO_DRIVE = True

DOTA_EXPERIMENT_RUNS.mkdir(parents=True, exist_ok=True)
DOTA_EXPERIMENT_ARTIFACTS.mkdir(parents=True, exist_ok=True)

print('TARGET_DATASET_ROOT =', TARGET_DATASET_ROOT)
print('SOURCE_DATASET_ROOT =', SOURCE_DATASET_ROOT)
print('SOURCE_ZIP_DIR =', SOURCE_ZIP_DIR)
print('DOTA_EXPERIMENT_ROOT =', DOTA_EXPERIMENT_ROOT)
print('DOTA_EXPERIMENT_RUNS =', DOTA_EXPERIMENT_RUNS)
print('DOTA_EXPERIMENT_ARTIFACTS =', DOTA_EXPERIMENT_ARTIFACTS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TARGET_DATASET_ROOT = /content/drive/MyDrive/datasets/dota_yolo
SOURCE_DATASET_ROOT = /content/drive/MyDrive/datasets/incoming/dota_yolo
SOURCE_ZIP_DIR = /content/drive/MyDrive/datasets/dota_zips


In [3]:
from pathlib import Path
import zipfile, shutil, os

DRIVE_ROOT = Path('/content/drive/MyDrive')

TARGET_DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'dota_yolo'
SOURCE_ZIP_DIR = DRIVE_ROOT / 'datasets' / 'dota_zips'

SOURCE_ZIP_DIR.mkdir(parents=True, exist_ok=True)
TARGET_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)

zip_path = SOURCE_ZIP_DIR / 'DOTAv1.zip'

# Готовый DOTA v1 в YOLO/Ultralytics формате
url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/DOTAv1.zip'

if not zip_path.exists():
    !wget -O "{zip_path}" "{url}"
else:
    print('[SKIP] ZIP already exists:', zip_path)

# Распаковка
tmp_extract = DRIVE_ROOT / 'datasets' / '_dota_extract_tmp'

if tmp_extract.exists():
    shutil.rmtree(tmp_extract)

tmp_extract.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(tmp_extract)

# Внутри архива обычно папка DOTAv1
candidates = [
    tmp_extract / 'DOTAv1',
    tmp_extract / 'dota1',
    tmp_extract,
]

src = None
for c in candidates:
    if (c / 'images').exists() and (c / 'labels').exists():
        src = c
        break

if src is None:
    raise RuntimeError('Не найдена YOLO-структура images/labels после распаковки')

if TARGET_DATASET_ROOT.exists():
    shutil.rmtree(TARGET_DATASET_ROOT)

shutil.copytree(src, TARGET_DATASET_ROOT)
shutil.rmtree(tmp_extract)

print('[OK] Dataset saved to:', TARGET_DATASET_ROOT)
print('train images:', len(list((TARGET_DATASET_ROOT / 'images' / 'train').glob('*'))))
print('val images:', len(list((TARGET_DATASET_ROOT / 'images' / 'val').glob('*'))))
print('train labels:', len(list((TARGET_DATASET_ROOT / 'labels' / 'train').glob('*'))))
print('val labels:', len(list((TARGET_DATASET_ROOT / 'labels' / 'val').glob('*'))))


[SKIP] ZIP already exists: /content/drive/MyDrive/datasets/dota_zips/DOTAv1.zip
[OK] Dataset saved to: /content/drive/MyDrive/datasets/dota_yolo
train images: 1411
val images: 458
train labels: 1411
val labels: 458


In [4]:
# Step 2: clone project repo if needed.
import subprocess
import sys

REPO_URL = 'https://github.com/s44w/small_objects_auto_aug.git'
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Using PROJECT_ROOT:', PROJECT_ROOT)


Using PROJECT_ROOT: /content/small_objects_auto_aug


In [5]:
# Step 3: copy prepared dataset to MyDrive target (idempotent and safe).
import shutil

FORCE_RECOPY = False
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}


def _count_images(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def _is_ready_yolo(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    if not all(p.exists() for p in req):
        return False
    return _count_images(root / 'images' / 'train') > 0 and _count_images(root / 'images' / 'val') > 0


src_resolved = SOURCE_DATASET_ROOT.resolve() if SOURCE_DATASET_ROOT.exists() else SOURCE_DATASET_ROOT
dst_resolved = TARGET_DATASET_ROOT.resolve() if TARGET_DATASET_ROOT.exists() else TARGET_DATASET_ROOT

if _is_ready_yolo(TARGET_DATASET_ROOT) and not FORCE_RECOPY:
    print('[SKIP] Target dataset already exists and looks ready.')
elif src_resolved == dst_resolved:
    # Source and target point to the same directory. Do not delete/copy over itself.
    if not _is_ready_yolo(TARGET_DATASET_ROOT):
        raise FileNotFoundError(
            f'SOURCE_DATASET_ROOT == TARGET_DATASET_ROOT but dataset is not ready: {TARGET_DATASET_ROOT}\n'
            'Either place a prepared YOLO dataset there, or run ZIP extraction step.'
        )
    print('[SKIP] Source and target are the same folder; dataset already in place.')
else:
    if not SOURCE_DATASET_ROOT.exists():
        raise FileNotFoundError(
            f'SOURCE_DATASET_ROOT not found: {SOURCE_DATASET_ROOT}\n'
            'Place prepared YOLO dataset there or use Step 4 to unpack ZIPs.'
        )

    if TARGET_DATASET_ROOT.exists():
        shutil.rmtree(TARGET_DATASET_ROOT)
    shutil.copytree(SOURCE_DATASET_ROOT, TARGET_DATASET_ROOT)
    print('[OK] Dataset copied to:', TARGET_DATASET_ROOT)

print('train images:', _count_images(TARGET_DATASET_ROOT / 'images' / 'train'))
print('val images:', _count_images(TARGET_DATASET_ROOT / 'images' / 'val'))


[SKIP] Target dataset already exists and looks ready.
train images: 1411
val images: 458


In [6]:
# Step 4 (optional): unpack ZIP dataset in place (if source is ZIPs).
# Use this step when SOURCE_ZIP_DIR contains archives and SOURCE_DATASET_ROOT is absent.
import zipfile
import shutil

RUN_UNZIP_FROM_SOURCE_ZIPS = False
FORCE_UNZIP = False

if RUN_UNZIP_FROM_SOURCE_ZIPS:
    if _is_ready_yolo(TARGET_DATASET_ROOT) and not FORCE_UNZIP:
        print('[SKIP] Target dataset already prepared. Set FORCE_UNZIP=True to rebuild from ZIPs.')
    else:
        if not SOURCE_ZIP_DIR.exists():
            raise FileNotFoundError(f'ZIP source dir not found: {SOURCE_ZIP_DIR}')

        zip_files = sorted(SOURCE_ZIP_DIR.glob('*.zip'))
        if not zip_files:
            raise FileNotFoundError(f'No .zip files in: {SOURCE_ZIP_DIR}')

        if TARGET_DATASET_ROOT.exists() and FORCE_UNZIP:
            shutil.rmtree(TARGET_DATASET_ROOT)
        TARGET_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

        for z in zip_files:
            print('[UNZIP]', z.name)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(TARGET_DATASET_ROOT)

        print('[OK] Unzip finished:', TARGET_DATASET_ROOT)
else:
    print('ZIP extraction skipped. Set RUN_UNZIP_FROM_SOURCE_ZIPS=True if needed.')


ZIP extraction skipped. Set RUN_UNZIP_FROM_SOURCE_ZIPS=True if needed.


In [7]:
# Step 5: build runtime config from template.
import yaml

base_cfg_path = PROJECT_ROOT / 'configs' / 'dota_config.yaml'
runtime_cfg_path = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_runtime_config.yaml'
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with base_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['kind'] = 'yolo_generic'
cfg['dataset']['mode'] = 'manual'
cfg['dataset']['root'] = str(TARGET_DATASET_ROOT)

# Persist training runs directly to Google Drive.
if isinstance(cfg.get('training'), dict):
    cfg['training']['project_dir'] = str(DOTA_EXPERIMENT_RUNS)
    cfg['training']['data_yaml'] = str(TARGET_DATASET_ROOT / 'dataset.yaml')
    cfg['training']['run_ablation'] = False
    cfg['training']['cache'] = 'disk'
    cfg['training']['val_during_train'] = True

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print('Runtime config:', runtime_cfg_path)
print('Training runs dir (Drive):', cfg['training']['project_dir'])


Runtime config: /content/small_objects_auto_aug/artifacts/dota_runtime_config.yaml


In [8]:
# Step 6: smoke run (subset -> analyze -> policy).
from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.utils.io import load_yaml

cfg = load_yaml(runtime_cfg_path)

subset_root = PROJECT_ROOT / 'datasets' / f'{DATASET_NAME.lower()}_smoke'
smoke_stats_dir = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_smoke' / 'stats'
smoke_policy_dir = PROJECT_ROOT / 'artifacts' / f'{DATASET_NAME.lower()}_smoke' / 'policy'

subset_report = build_yolo_subset(
    dataset_root=TARGET_DATASET_ROOT,
    output_root=subset_root,
    train_images=120,
    val_images=40,
    seed=42,
    clean_output=True,
)
print('subset report:', subset_report)

stats = analyze_dataset(
    dataset_root=subset_root,
    output_dir=smoke_stats_dir,
    splits=('train',),
    config=DatasetAnalyzerConfig(
        small_max_area=float(cfg['analysis']['small_max_area']),
        medium_max_area=float(cfg['analysis']['medium_max_area']),
        tiny_max_area=float(cfg['analysis']['tiny_max_area']),
        image_extensions=tuple(cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        generate_plots=False,
    ),
)

policy, decision_report = generate_policy_from_stats(
    stats,
    cfg=RuleEngineConfig.from_project_config(cfg),
)
paths = save_policy_artifacts(policy, decision_report, output_dir=smoke_policy_dir)

print('[OK] smoke done')
print('stats:', smoke_stats_dir / 'dataset_stats.json')
print('policy:', paths['policy_json'])


subset report: SubsetBuildReport(train_images=120, val_images=40, output_root='/content/small_objects_auto_aug/datasets/dota_smoke')


Analyze train:   0%|          | 0/120 [00:00<?, ?img/s]

[OK] smoke done
stats: /content/small_objects_auto_aug/artifacts/dota_smoke/stats/dataset_stats.json
policy: /content/small_objects_auto_aug/artifacts/dota_smoke/policy/policy_adaptive.json


In [9]:
# Step 7: full pipeline run with unified execution profiles.
from pathlib import Path
import shutil
import yaml
from src.pipeline_mvp import run_mvp_pipeline

assert (PROJECT_ROOT / 'configs' / 'baseline.yaml').exists(), 'Missing configs/baseline.yaml'
assert (PROJECT_ROOT / 'configs' / 'manual.yaml').exists(), 'Missing configs/manual.yaml'

# Optional speed-up: mirror dataset from Drive to local Colab SSD.
USE_LOCAL_RUNTIME_DATASET = True
LOCAL_DATASET_ROOT = Path('/content/datasets/dota_yolo')

if USE_LOCAL_RUNTIME_DATASET:
    train_marker = LOCAL_DATASET_ROOT / 'images' / 'train'
    val_marker = LOCAL_DATASET_ROOT / 'images' / 'val'
    if not (train_marker.exists() and val_marker.exists()):
        print(f'[INFO] Copy dataset to local runtime: {TARGET_DATASET_ROOT} -> {LOCAL_DATASET_ROOT}')
        if LOCAL_DATASET_ROOT.exists():
            shutil.rmtree(LOCAL_DATASET_ROOT)
        LOCAL_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(TARGET_DATASET_ROOT, LOCAL_DATASET_ROOT)
    else:
        print(f'[OK] Reuse local dataset: {LOCAL_DATASET_ROOT}')

# Unified profiles (similar to other notebooks):
# fast, balanced, hour_safe, hour_full, max_quality
EXECUTION_PROFILE = 'hour_safe'

profile_cfg = {
    'fast': {
        'run_training': True,
        'run_eval': False,
        'train_profile': 'fast',
        'run_ablation': False,
        'auto_predict_for_eval': False,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 8,
        'mode_overrides': {
            'baseline': {'epochs': 6, 'fraction': 0.2, 'imgsz': 768, 'val': True},
            'manual': {'epochs': 6, 'fraction': 0.2, 'imgsz': 768, 'val': True},
            'adaptive': {'epochs': 8, 'fraction': 0.3, 'imgsz': 768, 'val': True},
        },
    },
    'balanced': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'balanced',
        'run_ablation': False,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 10,
        'mode_overrides': {
            'baseline': {'epochs': 10, 'fraction': 0.4, 'imgsz': 896, 'val': True},
            'manual': {'epochs': 10, 'fraction': 0.4, 'imgsz': 896, 'val': True},
            'adaptive': {'epochs': 15, 'fraction': 0.8, 'imgsz': 896, 'val': True},
        },
    },
    'hour_safe': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'hour',
        'run_ablation': False,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 12,
        'mode_overrides': {
            'baseline': {'epochs': 10, 'fraction': 0.5, 'imgsz': 896, 'val': True},
            'manual': {'epochs': 10, 'fraction': 0.5, 'imgsz': 896, 'val': True},
            'adaptive': {'epochs': 24, 'fraction': 1.0, 'imgsz': 960, 'val': True},
        },
    },
    'hour_full': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'hour',
        'run_ablation': True,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 20,
        'mode_overrides': {},
    },
    'max_quality': {
        'run_training': True,
        'run_eval': True,
        'train_profile': 'max_quality',
        'run_ablation': True,
        'auto_predict_for_eval': True,
        'val_during_train': True,
        'cache': 'disk',
        'patience': 30,
        'mode_overrides': {},
    },
}

assert EXECUTION_PROFILE in profile_cfg, f'Unknown EXECUTION_PROFILE: {EXECUTION_PROFILE}'
selected = profile_cfg[EXECUTION_PROFILE]

# Update runtime config before run.
with runtime_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg.setdefault('dataset', {})
cfg.setdefault('training', {})

if USE_LOCAL_RUNTIME_DATASET:
    cfg['dataset']['root'] = str(LOCAL_DATASET_ROOT)

cfg['training']['run_ablation'] = bool(selected['run_ablation'])
cfg['training']['project_dir'] = str(DOTA_EXPERIMENT_RUNS)
cfg['training']['val_during_train'] = bool(selected['val_during_train'])
cfg['training']['cache'] = selected['cache']
cfg['training']['patience'] = int(selected['patience'])
cfg['training']['mode_overrides'] = selected['mode_overrides']

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

# Rough ETA estimate.
profile_epochs = {
    'fast': int(cfg['training'].get('epochs_fast', 10)),
    'final': int(cfg['training'].get('epochs_final', 80)),
    'balanced': 15,
    'quality': 25,
    'hour': 40,
    'max_quality': 60,
}
base_epochs = profile_epochs.get(selected['train_profile'], 40)
mode_ovr = selected.get('mode_overrides', {})
train_modes = ['baseline', 'manual', 'adaptive']
if selected['run_ablation']:
    train_modes += ['adaptive_no_mosaic', 'adaptive_no_custom_albu']

def _mode_epochs(mode_name: str) -> int:
    item = mode_ovr.get(mode_name, {}) if isinstance(mode_ovr, dict) else {}
    return int(item.get('epochs', base_epochs))

def _mode_val(mode_name: str) -> bool:
    item = mode_ovr.get(mode_name, {}) if isinstance(mode_ovr, dict) else {}
    return bool(item.get('val', selected['val_during_train']))

train_epoch_minutes = 3.8
val_epoch_minutes = 0.45
post_eval_minutes = 2.0 if selected['run_eval'] else 0.0
eta_min = 0.0
for m in train_modes:
    e = _mode_epochs(m)
    eta_min += e * train_epoch_minutes
    if _mode_val(m):
        eta_min += e * val_epoch_minutes
if selected['run_eval']:
    eta_min += len(train_modes) * post_eval_minutes

print(f'[INFO] EXECUTION_PROFILE={EXECUTION_PROFILE}')
print(f'[INFO] train_profile={selected["train_profile"]}, run_ablation={selected["run_ablation"]}')
print(f'[INFO] train modes={train_modes}')
print(f'[INFO] estimated time ~ {eta_min:.0f} min (~{eta_min/60:.1f} h)')
print(f'[INFO] training.project_dir={cfg["training"]["project_dir"]}')
print(f'[INFO] dataset.root={cfg["dataset"]["root"]}')

outputs = run_mvp_pipeline(
    project_config_path=runtime_cfg_path,
    run_training=bool(selected['run_training']),
    run_eval=bool(selected['run_eval']),
    train_profile=str(selected['train_profile']),
    auto_predict_for_eval=bool(selected['auto_predict_for_eval']),
)
print(outputs)

# Persist artifacts to Drive (runs are already written to DOTA_EXPERIMENT_RUNS).
if SYNC_ARTIFACTS_TO_DRIVE:
    local_artifacts = PROJECT_ROOT / 'artifacts'
    if local_artifacts.exists():
        shutil.copytree(local_artifacts, DOTA_EXPERIMENT_ARTIFACTS, dirs_exist_ok=True)
    print('[OK] Artifacts synced to:', DOTA_EXPERIMENT_ARTIFACTS)

manifest_path = Path(outputs.get('experiment_manifest', ''))
if manifest_path.exists() and SYNC_ARTIFACTS_TO_DRIVE:
    manifest_copy = DOTA_EXPERIMENT_ARTIFACTS / 'experiment_manifest.json'
    shutil.copy2(manifest_path, manifest_copy)
    print('[OK] Manifest copied:', manifest_copy)


Analyze train:   0%|          | 0/1411 [00:00<?, ?img/s]

Analyze val:   0%|          | 0/458 [00:00<?, ?img/s]

FileNotFoundError: [Errno 2] No such file or directory: 'configs/baseline.yaml'

In [ ]:
# Step X: unified monitoring export (train + val metrics history).
from pathlib import Path
import shutil
import pandas as pd

# This cell builds reusable CSV artifacts for later plots in thesis:
# 1) full epoch history across runs
# 2) last-epoch snapshot per run
# It includes both train-side metrics (losses) and val-side metrics (precision/recall/mAP).

PROJECT_ROOT_RESOLVED = Path(globals().get('PROJECT_ROOT', Path.cwd()))


def _resolve_runtime_cfg_path() -> Path | None:
    candidate_names = ['runtime_cfg_path']
    for name in candidate_names:
        value = globals().get(name)
        if value is not None:
            p = Path(value)
            if p.exists():
                return p

    hints = [
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'dota_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'xview_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'coco_small_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'dota_scaleaware_vs_adaptive_runtime.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'runtime_config.yaml',
    ]
    for p in hints:
        if p.exists():
            return p
    return None


def _load_cfg(cfg_path: Path | None):
    if cfg_path is None:
        return None
    try:
        import yaml
        return yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
    except Exception:
        return None


def _resolve_runs_root(cfg: dict | None) -> Path:
    name_candidates = [
        'RUNS_ROOT',
        'RUNS_DRIVE_DIR',
        'DOTA_EXPERIMENT_RUNS',
        'COCO_EXPERIMENT_RUNS',
        'XVIEW_EXPERIMENT_RUNS',
    ]
    for name in name_candidates:
        value = globals().get(name)
        if value is not None:
            p = Path(value)
            if p.exists():
                return p

    if isinstance(cfg, dict):
        p = cfg.get('training', {}).get('project_dir')
        if p:
            rp = Path(p)
            if rp.exists():
                return rp

    fallback = PROJECT_ROOT_RESOLVED / 'runs'
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


def _resolve_drive_artifacts_root() -> Path | None:
    candidates = [
        globals().get('ARTIFACTS_ROOT'),
        globals().get('DOTA_EXPERIMENT_ARTIFACTS'),
        globals().get('COCO_EXPERIMENT_ARTIFACTS'),
        globals().get('XVIEW_EXPERIMENT_ARTIFACTS'),
        globals().get('ARTIFACTS_DRIVE_DIR'),
        globals().get('DRIVE_ARTIFACTS_ROOT'),
    ]
    for item in candidates:
        if item is None:
            continue
        p = Path(item)
        try:
            p.mkdir(parents=True, exist_ok=True)
            return p
        except Exception:
            continue
    return None


cfg_path = _resolve_runtime_cfg_path()
cfg = _load_cfg(cfg_path)
runs_root = _resolve_runs_root(cfg)

monitor_dir = PROJECT_ROOT_RESOLVED / 'artifacts' / 'monitoring'
monitor_dir.mkdir(parents=True, exist_ok=True)

results_files = sorted(runs_root.rglob('results.csv'))
if not results_files:
    raise FileNotFoundError(f'No results.csv found under runs root: {runs_root}')

frames = []
for csv_path in results_files:
    run_dir = csv_path.parent
    run_name = run_dir.name
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f'[WARN] failed to read {csv_path}: {exc}')
        continue
    if df.empty:
        continue

    if 'epoch' not in df.columns:
        df['epoch'] = list(range(len(df)))

    df['run_name'] = run_name
    df['run_dir'] = run_dir.as_posix()
    frames.append(df)

if not frames:
    raise RuntimeError('No readable non-empty results.csv files were found.')

history_wide = pd.concat(frames, ignore_index=True)

# Keep numeric metric columns + id columns.
id_cols = ['run_name', 'run_dir', 'epoch']
metric_cols = [
    c for c in history_wide.columns
    if c not in id_cols and pd.api.types.is_numeric_dtype(history_wide[c])
]
history_wide = history_wide[id_cols + metric_cols]

history_wide_path = monitor_dir / 'metrics_history_wide.csv'
history_wide.to_csv(history_wide_path, index=False)

# Last-epoch snapshot per run (useful for quick comparison tables).
last_rows = history_wide.sort_values(['run_name', 'epoch']).groupby('run_name', as_index=False).tail(1)
last_rows = last_rows.sort_values('run_name').reset_index(drop=True)
last_rows_path = monitor_dir / 'metrics_last_epoch.csv'
last_rows.to_csv(last_rows_path, index=False)

# Build long-format table for plotting (run/epoch/metric/value).
history_long = history_wide.melt(
    id_vars=['run_name', 'run_dir', 'epoch'],
    value_vars=metric_cols,
    var_name='metric',
    value_name='value',
)

def _metric_split(metric_name: str) -> str:
    m = metric_name.lower()
    if m.startswith('metrics/') or m.startswith('val/'):
        return 'val'
    if m.startswith('train/') or m.endswith('_loss'):
        return 'train'
    if 'loss' in m:
        return 'train'
    return 'other'

history_long['split'] = history_long['metric'].map(_metric_split)
history_long_path = monitor_dir / 'metrics_history_long.csv'
history_long.to_csv(history_long_path, index=False)

# Quick console-friendly view for thesis workflow.
interesting = [
    'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
    'box_loss', 'cls_loss', 'dfl_loss',
    'metrics/precision(B)', 'metrics/recall(B)',
    'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
]
show_cols = ['run_name', 'epoch'] + [c for c in interesting if c in last_rows.columns]
print('[OK] history_wide:', history_wide_path)
print('[OK] history_long:', history_long_path)
print('[OK] last_epoch:', last_rows_path)
print('\nLast-epoch snapshot (train + val):')
print(last_rows[show_cols].to_string(index=False))

# Optional Drive sync for artifacts.
drive_artifacts_root = _resolve_drive_artifacts_root()
if drive_artifacts_root is not None:
    drive_monitor_dir = drive_artifacts_root / 'monitoring'
    drive_monitor_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(history_wide_path, drive_monitor_dir / history_wide_path.name)
    shutil.copy2(history_long_path, drive_monitor_dir / history_long_path.name)
    shutil.copy2(last_rows_path, drive_monitor_dir / last_rows_path.name)
    print('[OK] Monitoring CSVs synced to Drive:', drive_monitor_dir)


In [ ]:
# Step Y: plot preview for train/val metrics (for thesis figures draft).
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt

RUN_PLOT_PREVIEW = True

if RUN_PLOT_PREVIEW:
    project_root = Path(globals().get('PROJECT_ROOT', Path.cwd()))
    monitor_dir = project_root / 'artifacts' / 'monitoring'
    history_long_path = monitor_dir / 'metrics_history_long.csv'

    if not history_long_path.exists():
        raise FileNotFoundError(
            f'Missing monitoring history: {history_long_path}. '
            'Run the monitoring export cell first.'
        )

    df = pd.read_csv(history_long_path)
    if df.empty:
        raise RuntimeError('metrics_history_long.csv is empty.')

    # Keep numeric rows only.
    df = df[pd.to_numeric(df['value'], errors='coerce').notna()].copy()
    df['value'] = df['value'].astype(float)

    plots_dir = monitor_dir / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Metrics that are usually useful for diploma report.
    metric_candidates = [
        'metrics/mAP50-95(B)',
        'metrics/mAP50(B)',
        'metrics/precision(B)',
        'metrics/recall(B)',
        'train/box_loss',
        'train/cls_loss',
        'train/dfl_loss',
        'box_loss',
        'cls_loss',
        'dfl_loss',
    ]
    metrics = [m for m in metric_candidates if m in set(df['metric'].unique())]

    if not metrics:
        raise RuntimeError('No known metrics found for plotting in metrics_history_long.csv.')

    created = []

    # 1) Per-metric comparison across runs.
    for metric in metrics:
        sub = df[df['metric'] == metric].copy()
        if sub.empty:
            continue

        plt.figure(figsize=(9, 5))
        for run_name, grp in sub.groupby('run_name'):
            grp = grp.sort_values('epoch')
            plt.plot(grp['epoch'], grp['value'], marker='o', markersize=2, linewidth=1.6, label=str(run_name))

        plt.title(f'{metric} vs epoch')
        plt.xlabel('epoch')
        plt.ylabel(metric)
        plt.grid(alpha=0.3)
        plt.legend(loc='best', fontsize=8)
        out = plots_dir / f"metric_{metric.replace('/', '_').replace('(', '').replace(')', '')}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.close()
        created.append(out)

    # 2) Overview: best val metrics by run (last epoch).
    last_rows = (
        df.sort_values(['run_name', 'epoch'])
          .groupby(['run_name', 'metric'], as_index=False)
          .tail(1)
    )

    val_metrics_for_bar = [m for m in ['metrics/mAP50-95(B)', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)'] if m in set(last_rows['metric'])]
    if val_metrics_for_bar:
        pivot = (
            last_rows[last_rows['metric'].isin(val_metrics_for_bar)]
            .pivot_table(index='run_name', columns='metric', values='value', aggfunc='mean')
            .sort_index()
        )

        ax = pivot.plot(kind='bar', figsize=(10, 5), rot=25)
        ax.set_title('Final val metrics by run (last epoch)')
        ax.set_ylabel('value')
        ax.grid(axis='y', alpha=0.3)
        out_bar = plots_dir / 'final_val_metrics_bar.png'
        plt.tight_layout()
        plt.savefig(out_bar, dpi=160)
        plt.close()
        created.append(out_bar)

    print('[OK] Plot files created:')
    for p in created:
        print(' -', p)

    # Optional Drive sync.
    drive_candidates = [
        globals().get('ARTIFACTS_ROOT'),
        globals().get('DOTA_EXPERIMENT_ARTIFACTS'),
        globals().get('COCO_EXPERIMENT_ARTIFACTS'),
        globals().get('XVIEW_EXPERIMENT_ARTIFACTS'),
        globals().get('ARTIFACTS_DRIVE_DIR'),
        globals().get('DRIVE_ARTIFACTS_ROOT'),
    ]
    drive_root = None
    for c in drive_candidates:
        if c is None:
            continue
        try:
            p = Path(c)
            p.mkdir(parents=True, exist_ok=True)
            drive_root = p
            break
        except Exception:
            continue

    if drive_root is not None:
        drive_plots_dir = drive_root / 'monitoring' / 'plots'
        drive_plots_dir.mkdir(parents=True, exist_ok=True)
        for p in created:
            shutil.copy2(p, drive_plots_dir / p.name)
        print('[OK] Plot files synced to Drive:', drive_plots_dir)
else:
    print('Plot preview skipped. Set RUN_PLOT_PREVIEW=True to enable.')
